# Customer Segmentation for an Online Retailer using Clustering

**MIS 451 – Machine Learning for Business** · Group Project: *Clustering – Business Pattern Discovery and Segmentation*

| Member | Role | Sections |
|---|---|---|
| Dương (Team leader) | Data Engineer | 1, 2, 3, 5, 6, 9 (Julius AI) |
| Như | EDA & Report | 4, 11 |
| Ngọc | ML Modeler | 7, 8, 10 |
| Bảo | Tools & Business Insight | 9, 10 |

**Dataset:** Online Retail II – UCI Machine Learning Repository

## Table of Contents
1. Business Context & Objectives
2. Data Collection
3. Setup & Load Data
4. Exploratory Data Analysis (EDA)
5. Data Cleaning
6. Feature Engineering & Transformation
7. Model Development
8. Model Evaluation & Comparison
9. Tool Comparison: Python vs Orange vs Julius AI
10. Cluster Profiling & Business Recommendations
11. Limitations & Future Work
12. Team Member Roles

> **Team rule:** each member only edits their own sections. Run **Runtime → Run all** before starting so all variables are available.

---
## 1. Business Context & Objectives
*Owner: Dương*

**Context.** The company is a UK-based online retailer selling unique all-occasion gifts. Many of its customers are wholesalers. The business has over one million transactions but treats all customers the same way in its marketing.

**Problem.** Without knowing which customers are valuable, loyal, new or about to leave, marketing budget is spent evenly and high-value customers may churn unnoticed.

**Objective.** Use unsupervised learning (clustering) to group customers by their purchasing behaviour, then turn each segment into a concrete marketing action.

**Key questions**
- What distinct customer segments exist?
- Which segments generate most of the revenue?
- Which customers are at risk of leaving, and how can we win them back?

---
## 2. Data Collection
*Owner: Dương*

- **Source:** UCI Machine Learning Repository – Online Retail II (https://archive.ics.uci.edu/dataset/502/online+retail+ii)
- **Period:** 01/12/2009 – 09/12/2011
- **Size:** 1,067,371 transactions, 8 columns
- **Original format:** Excel file with 2 sheets (2009–2010 and 2010–2011). We combined both sheets into one compressed CSV (`data/online_retail_II.csv.gz`) to fit GitHub's file size limit.

| Column | Description |
|---|---|
| Invoice | Invoice number. Starts with "C" if the order was cancelled |
| StockCode | Product code |
| Description | Product name |
| Quantity | Quantity of the product in the invoice |
| InvoiceDate | Date and time of the invoice |
| Price | Unit price (£) |
| Customer ID | Unique customer number |
| Country | Customer's country |

Each row is **one product line in one invoice**. To segment customers, we will later turn it into **one row per customer**.

---
## 3. Setup & Load Data
*Owner: Dương*

The data is read directly from our GitHub repo, so no file upload is needed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
path = 'https://raw.githubusercontent.com/DuongvuBBS20/MIS451-Customer-Segmentation/main/data/online_retail_II.csv.gz'

df = pd.read_csv(path, parse_dates=['InvoiceDate'], dtype={'Invoice': str, 'StockCode': str})

print('Rows:', len(df))
df.head()

In [ ]:
df.info()

In [ ]:
# Keep an untouched copy of the raw data for EDA (Section 4)
df_raw = df.copy()

---
## 4. Exploratory Data Analysis (EDA)
*Owner: Như*

Use `df_raw` (the raw data) for this section.

Suggested analysis:
- Missing values per column
- Distribution of Quantity and Price
- Revenue by month (seasonality)
- Revenue / customers by country
- Top products and top customers

In [ ]:
# TODO (Như): EDA on df_raw

---
## 5. Data Cleaning
*Owner: Dương*

**5.1 Remove rows without Customer ID** – we segment customers, so a row without a customer cannot be used.

In [ ]:
df = df.dropna(subset=['Customer ID'])
df['Customer ID'] = df['Customer ID'].astype(int)
print('Rows:', len(df))

**5.2 Remove duplicate rows** – the two original sheets overlap (1–9 Dec 2010), so some transactions appear twice.

In [ ]:
df = df.drop_duplicates()
print('Rows:', len(df))

**5.3 Remove cancelled orders** – invoice numbers starting with "C" are cancellations, not purchases.

In [ ]:
df = df[~df['Invoice'].str.startswith('C')]
print('Rows:', len(df))

**5.4 Keep only positive quantity and price.**

In [ ]:
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
print('Rows:', len(df))

**5.5 Remove codes that are not products** – e.g. POST (postage), M (manual), BANK CHARGES. Real product codes start with a number.

In [ ]:
df = df[df['StockCode'].str[0].str.isdigit()]
print('Rows:', len(df))

**5.6 Add total price of each line.**

In [ ]:
df['TotalPrice'] = df['Quantity'] * df['Price']

**Cleaning summary**

| Step | Rows remaining |
|---|---|
| Raw data | 1,067,371 |
| Remove missing Customer ID | 824,364 |
| Remove duplicates | 797,885 |
| Remove cancelled orders | 779,495 |
| Keep Quantity > 0 and Price > 0 | 779,425 |
| Remove non-product codes | 776,577 |

---
## 6. Feature Engineering & Transformation
*Owner: Dương*

**6.1 Create 7 features for each customer**

| Feature | Meaning | Data type |
|---|---|---|
| Recency | Days since last purchase | Date/time |
| Frequency | Number of orders | Numeric |
| Monetary | Total money spent (£) | Numeric |
| AvgOrderValue | Money per order | Numeric |
| UniqueProducts | Number of different products bought | Numeric |
| Tenure | Days since first purchase | Date/time |
| IsUK | 1 = customer in the UK, 0 = other country | Categorical |

In [ ]:
# "Today" = 1 day after the last date in the data
today = df['InvoiceDate'].max() + pd.Timedelta(days=1)

customers = df.groupby('Customer ID').agg(
    LastPurchase=('InvoiceDate', 'max'),
    FirstPurchase=('InvoiceDate', 'min'),
    Frequency=('Invoice', 'nunique'),
    Monetary=('TotalPrice', 'sum'),
    UniqueProducts=('StockCode', 'nunique'),
    Country=('Country', 'first'),
)

customers['Recency'] = (today - customers['LastPurchase']).dt.days
customers['Tenure'] = (today - customers['FirstPurchase']).dt.days
customers['AvgOrderValue'] = customers['Monetary'] / customers['Frequency']
customers['IsUK'] = (customers['Country'] == 'United Kingdom').astype(int)

features = ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'UniqueProducts', 'Tenure', 'IsUK']
customers = customers[features]

print('Number of customers:', len(customers))
customers.describe()

**6.2 Fix skewed data** – a few customers spend extremely large amounts, so the data is very skewed. We use log to make it more balanced.

In [ ]:
customers.hist(figsize=(12, 8), bins=50);

In [ ]:
data = customers.copy()

skewed = ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'UniqueProducts']
for col in skewed:
    data[col] = np.log1p(data[col])

data.hist(figsize=(12, 8), bins=50);

**6.3 Scale the data** – the features use different units (days, pounds, counts). StandardScaler puts them on the same scale so no feature dominates the clustering.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)
data_scaled = pd.DataFrame(data_scaled, index=data.index, columns=features)

data_scaled.head()

**Variables ready for the next sections**

| Variable | Content | Use it for |
|---|---|---|
| `data_scaled` | Scaled features (one row per customer) | **Training the models** (Section 7) |
| `customers` | Original numbers (days, £, counts) | **Explaining the clusters** (Section 10) |

> Note: about 90% of customers are in the UK. Try the models **with and without `IsUK`** to check it does not dominate the clusters.

**6.4 (Optional) Download the CSV files** – only needed for Orange and Julius AI (Section 9).

In [ ]:
# data_scaled.to_csv('customer_features.csv')
# customers.to_csv('customer_features_raw.csv')
# from google.colab import files
# files.download('customer_features.csv')
# files.download('customer_features_raw.csv')

---
## 7. Model Development
*Owner: Ngọc*

Input: `data_scaled`

### 7.1 K-Means
Choose k with the Elbow method and Silhouette score.

In [ ]:
# TODO (Ngọc): K-Means

### 7.2 Hierarchical Clustering
Use Ward linkage and a dendrogram to choose the number of clusters.

In [ ]:
# TODO (Ngọc): Hierarchical clustering

### 7.3 Autoencoder + K-Means (Deep Learning)
Compress the features with an autoencoder, then run K-Means on the compressed data.

In [ ]:
# TODO (Ngọc): Autoencoder + K-Means

---
## 8. Model Evaluation & Comparison
*Owner: Ngọc*

Compare the models with Silhouette score (higher is better), Davies–Bouldin index (lower is better) and Calinski–Harabasz score (higher is better). Visualise the clusters with PCA and choose the best model.

In [ ]:
# TODO (Ngọc): comparison table + PCA plot

---
## 9. Tool Comparison: Python vs Orange vs Julius AI
*Owners: Bảo (Orange, comparison), Dương (Julius AI)*

All three tools use the same file: `customer_features.csv` (Section 6.4).

### 9.1 Orange
*(Bảo – add screenshots of the workflow and results)*

### 9.2 Julius AI
*(Dương – add the prompt used and screenshots of the results)*

### 9.3 Comparison
| Criteria | Python | Orange | Julius AI |
|---|---|---|---|
| How it works | Write code | Drag and drop | Type a prompt |
| Number of clusters / Silhouette | | | |
| Same result as Python? | (baseline) | | |
| Customisation | High | Medium | Low |
| Reproducibility | High | High (.ows file) | Low |
| Deep learning support | Yes | Limited | Depends on prompt |
| Cost | Free | Free | Free (limited) |
| Best for | Main analysis | Quick visual exploration | Non-coders |

**Why Python is our main tool:** *(Bảo – write conclusion)*

---
## 10. Cluster Profiling & Business Recommendations
*Owners: Bảo + Ngọc*

Use `customers` (original numbers) with the cluster labels from the best model.

In [ ]:
# TODO (Bảo + Ngọc): average features per cluster, cluster size, share of revenue

| Cluster | Name | Description | Recommended action |
|---|---|---|---|
| 0 | | | |
| 1 | | | |
| 2 | | | |
| 3 | | | |

---
## 11. Limitations & Future Work
*Owner: Như*

**Limitations**
- *(to be written)*

**Future work**
- *(to be written)*

---
## 12. Team Member Roles

| Member | Role | Tasks |
|---|---|---|
| Dương (Team leader) | Data Engineer | Business context, data collection, data cleaning, feature engineering, Julius AI, slides, GitHub |
| Như | EDA & Report | Exploratory data analysis, limitations & future work, final report |
| Ngọc | ML Modeler | K-Means, Hierarchical, Autoencoder, model evaluation, cluster naming |
| Bảo | Tools & Business Insight | Orange, tool comparison, cluster profiling, business recommendations |